# PSMAC NPU 4×4 — Pneumonia Detection Pipeline
### Download → Train → Quantize → Export hex (16 tiles) → Visualize

**Architecture: npu_4_4 → 4×4 array of psmac_top tiles, sequenced by npu_ctrl FSM**

- **16 tiles** (ROWS=4, COLS=4), each gets its own `weights_tile_RR_CC.hex`
- **4 activation columns**, each gets its own `input_col_CC_NN.hex`
- npu_ctrl LOAD_W phase: loops tile_sel 0→15, each tile gets `vec_len` weights
- npu_ctrl LOAD_X phase: loops col_sel 0→3, each column gets `vec_len` activations

**Run cells one by one. Wait for each to finish.**

In [ ]:
# ── CELL 1: Kaggle setup + download dataset ──────────────────────────────────
import os

# ↓↓↓ FILL THESE IN ↓↓↓
KAGGLE_USERNAME = "your_username_here"
KAGGLE_KEY      = "your_token_here"
# ↑↑↑ FILL THESE IN ↑↑↑

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    f.write('{"username":"' + KAGGLE_USERNAME + '","key":"' + KAGGLE_KEY + '"}')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

!pip install -q kaggle
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia
!unzip -q chest-xray-pneumonia.zip -d chest_xray
print('Dataset ready!')

In [ ]:
# ── CELL 2: Imports + helpers ─────────────────────────────────────────────────
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pathlib, random, glob, os, zipfile
from PIL import Image

IMG_SIZE = 64

# ── npu_4_4 topology constants ─────────────────────────────────────────────
NPU_ROWS   = 4
NPU_COLS   = 4
NPU_TILES  = NPU_ROWS * NPU_COLS   # 16
# vec_len in hardware is the penultimate layer width / NPU_ROWS
# (each row of tiles handles one slice of the activation vector)
# Actual split is determined after model is trained — recomputed in Cell 4

# ── Dataset auto-detect ────────────────────────────────────────────────────
for candidate in [
    '/content/chest_xray/chest_xray/train',
    '/content/chest_xray/train',
]:
    if os.path.exists(candidate):
        TRAIN_DIR = candidate
        TEST_DIR  = candidate.replace('train','test')
        break
else:
    raise FileNotFoundError("Dataset not found — re-run Cell 1")

print(f'Train: {TRAIN_DIR}')
print(f'Test:  {TEST_DIR}')

# ── Utility functions ──────────────────────────────────────────────────────
def find_images(folder):
    imgs  = glob.glob(f'{folder}/**/*.jpeg', recursive=True)
    imgs += glob.glob(f'{folder}/**/*.jpg',  recursive=True)
    imgs += glob.glob(f'{folder}/**/*.png',  recursive=True)
    return sorted(imgs)

def load_image(path, size=IMG_SIZE):
    img = Image.open(path).convert('L').resize((size, size))
    return np.array(img, dtype=np.float32) / 255.0

def to_hex(arr, fname):
    """Write int8 array to hex file (one byte per line, 2 hex digits)."""
    with open(fname, 'w') as f:
        for v in arr:
            f.write(f'{int(v) & 0xFF:02X}\n')

def quantize_array(arr, bits=8):
    """Symmetric min-max INT8 quantization."""
    max_val = max(abs(arr.max()), abs(arr.min()))
    scale   = max_val / 127.0 if max_val > 0 else 1.0
    return np.clip(np.round(arr / scale), -127, 127).astype(np.int8)

def dot_int8(a, w):
    return int(np.sum(a.astype(np.int32) * w.astype(np.int32)))

def build_penultimate(mdl, input_shape):
    inp = keras.Input(shape=input_shape)
    x   = inp
    for layer in mdl.layers[:-1]:
        x = layer(x)
    return keras.Model(inputs=inp, outputs=x)

n_tr = find_images(f'{TRAIN_DIR}/NORMAL')
p_tr = find_images(f'{TRAIN_DIR}/PNEUMONIA')
print(f'Train — NORMAL: {len(n_tr)}, PNEUMONIA: {len(p_tr)}')
print('Ready!')

In [ ]:
# ── CELL 3: Train whole-image classification model ────────────────────────────
# Dense(32) penultimate layer → 32 activations
# 32 activations split across 4 columns → 8 per column (col vec_len = 8)
# Each column feeds 4 rows  → tile vec_len = 8 (same)
# Final Dense(1) weight matrix: shape (32, 1) → split to 16 tiles (2 weights each)

BATCH  = 32
EPOCHS = 25

normal_files    = find_images(f'{TRAIN_DIR}/NORMAL')
pneumonia_files = find_images(f'{TRAIN_DIR}/PNEUMONIA')
random.seed(42)
random.shuffle(pneumonia_files)
pneumonia_files = pneumonia_files[:len(normal_files)]
print(f'Using {len(normal_files)} NORMAL + {len(pneumonia_files)} PNEUMONIA')

X, y = [], []
for f in normal_files:
    try: X.append(load_image(f)); y.append(0)
    except: pass
for f in pneumonia_files:
    try: X.append(load_image(f)); y.append(1)
    except: pass

X = np.array(X).reshape(-1, IMG_SIZE, IMG_SIZE, 1)
y = np.array(y, dtype=np.float32)
idx = np.random.permutation(len(X))
X, y = X[idx], y[idx]

Xt, yt = [], []
for f in find_images(f'{TEST_DIR}/NORMAL'):
    try: Xt.append(load_image(f)); yt.append(0)
    except: pass
for f in find_images(f'{TEST_DIR}/PNEUMONIA'):
    try: Xt.append(load_image(f)); yt.append(1)
    except: pass
Xt = np.array(Xt).reshape(-1, IMG_SIZE, IMG_SIZE, 1)
yt = np.array(yt, dtype=np.float32)
print(f'Train: {X.shape} | Test: {Xt.shape}')

model = keras.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1)),
    layers.Conv2D(8,  3, activation='relu', padding='same'),
    layers.MaxPooling2D(4),
    layers.Conv2D(16, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(4),
    layers.Flatten(),
    layers.Dense(32, activation='relu'),   # ← penultimate: 32-D
    layers.Dense(1,  activation='sigmoid') # ← final: weight shape (32,1)
], name='whole_image_cnn')

model.compile(
    optimizer=keras.optimizers.Adam(3e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)
model.summary()

cb = keras.callbacks.EarlyStopping(
    monitor='val_auc', patience=6,
    restore_best_weights=True, mode='max'
)

split = int(0.9 * len(X))
history = model.fit(
    X[:split], y[:split],
    batch_size=BATCH, epochs=EPOCHS,
    validation_data=(X[split:], y[split:]),
    callbacks=[cb]
)

preds = (model.predict(Xt, verbose=0) > 0.5).flatten()
acc_n = (preds[yt==0] == 0).mean() * 100
acc_p = (preds[yt==1] == 1).mean() * 100
print(f'\nNORMAL    accuracy: {acc_n:.0f}%')
print(f'PNEUMONIA accuracy: {acc_p:.0f}%')
print(f'Overall   accuracy: {((preds==yt).mean()*100):.0f}%')

In [ ]:
# ── CELL 4: Compute npu_4_4 tiling parameters ─────────────────────────────────
#
# npu_4_4 layout (tile index = row*4 + col):
#
#          col0   col1   col2   col3
#  row0  [ t00 ][ t01 ][ t02 ][ t03 ]  ← activations[0:8]   × weights[0:8]
#  row1  [ t04 ][ t05 ][ t06 ][ t07 ]  ← activations[8:16]  × weights[8:16]
#  row2  [ t08 ][ t09 ][ t10 ][ t11 ]  ← activations[16:24] × weights[16:24]
#  row3  [ t12 ][ t13 ][ t14 ][ t15 ]  ← activations[24:32] × weights[24:32]
#
# All 4 tiles in the same row share the same activation slice (same col_sel).
# Since there is only 1 output neuron, all 4 columns within a row receive
# the SAME weight slice — their results are summed in software (or in hw via
# the accumulator already summing across vec_len). The 16-tile dot products are
# added together to get the final scalar.
#
# penultimate layer output: shape (32,)  → split into 4 rows of 8 each
# final Dense weight:       shape (32,1) → split into 4 rows of 8 each
#
# vec_len for each tile = 8  (npu_ctrl counts 0→7 per tile/column)

ACT_DIM   = 32          # Dense(32) penultimate output
VEC_LEN   = ACT_DIM // NPU_ROWS   # = 8  → send to vec_len port

W_all = model.layers[-1].get_weights()[0].flatten()   # shape (32,)
print(f'Weight vector length  : {len(W_all)}')
print(f'ACT_DIM (penultimate) : {ACT_DIM}')
print(f'NPU_ROWS × NPU_COLS   : {NPU_ROWS} × {NPU_COLS}')
print(f'VEC_LEN per tile      : {VEC_LEN}')
print(f'\nExpected npu_ctrl FSM behaviour:')
print(f'  LOAD_W : tile_sel 0→15,  load_addr 0→{VEC_LEN-1}  ({NPU_TILES}×{VEC_LEN} = {NPU_TILES*VEC_LEN} write cycles)')
print(f'  LOAD_X : col_sel  0→3,   load_addr 0→{VEC_LEN-1}  ({NPU_COLS }×{VEC_LEN} = {NPU_COLS*VEC_LEN } write cycles)')
print(f'  RUN    : psmac_en=1 for 1 cycle → each psmac_top FSM runs for {VEC_LEN} cycles')

In [ ]:
# ── CELL 5: Export hex files for npu_4_4 ─────────────────────────────────────
#
# File naming follows npu_ctrl addressing:
#   weights_tile_{IDX:02d}.hex  — tile IDX = row*4+col, addr 0→VEC_LEN-1
#   input_col{CC}_sample{NN:02d}.hex — col CC = 0..3, sample NN = 0..39
#   labels.csv, metrics.txt

os.makedirs('psmac_data', exist_ok=True)

# ── Quantize and split weights across 16 tiles ───────────────────────────────
# W_all shape: (32,)  — row r gets W_all[r*VEC_LEN : (r+1)*VEC_LEN]
# All 4 cols in a row share the same weight slice (1 output neuron)
W_q = quantize_array(W_all)   # global INT8 quant

for tile_idx in range(NPU_TILES):
    row = tile_idx // NPU_COLS
    col = tile_idx  % NPU_COLS
    w_slice = W_q[row * VEC_LEN : (row+1) * VEC_LEN]
    to_hex(w_slice, f'psmac_data/weights_tile_{tile_idx:02d}.hex')

print(f'Exported {NPU_TILES} weight hex files  (each {VEC_LEN} bytes)')

# ── Build penultimate model ───────────────────────────────────────────────────
penultimate = build_penultimate(model, (IMG_SIZE, IMG_SIZE, 1))

# ── Collect test samples ──────────────────────────────────────────────────────
test_normal_files    = find_images(f'{TEST_DIR}/NORMAL')
test_pneumonia_files = find_images(f'{TEST_DIR}/PNEUMONIA')

def get_correct_samples(files, true_label, n=20):
    results = []
    for f in files:
        if len(results) >= n: break
        try:
            img  = load_image(f).reshape(1, IMG_SIZE, IMG_SIZE, 1)
            pred = float(model.predict(img, verbose=0)[0][0])
            correct = (true_label==1 and pred>0.5) or (true_label==0 and pred<=0.5)
            if correct:
                act   = penultimate.predict(img, verbose=0).flatten()  # shape (32,)
                act_q = quantize_array(act)
                # Compute tile-level dot products and sum (mimics 4×4 hardware)
                tile_dots = []
                for r in range(NPU_ROWS):
                    a_slice = act_q[r*VEC_LEN:(r+1)*VEC_LEN]
                    w_slice = W_q [r*VEC_LEN:(r+1)*VEC_LEN]
                    tile_dots.append(dot_int8(a_slice, w_slice))
                total_dot = sum(tile_dots)
                results.append({
                    'path': f, 'act_q': act_q,
                    'tile_dots': tile_dots,
                    'total_dot': total_dot,
                    'pred': pred, 'label': true_label
                })
        except: pass
    return results

print('Finding correctly classified test images...')
normal_samples    = get_correct_samples(test_normal_files,    0, 20)
pneumonia_samples = get_correct_samples(test_pneumonia_files, 1, 20)
all_samples       = pneumonia_samples + normal_samples
print(f'Got {len(pneumonia_samples)} PNEUMONIA + {len(normal_samples)} NORMAL samples')

# ── Export per-column activation hex + xray thumbnails ───────────────────────
# npu_ctrl LOAD_X: col_sel 0→3, each column gets VEC_LEN activations
# All tiles in a row share the same activation vector slice (same col_sel)
# col CC maps to activation slice [CC*VEC_LEN : (CC+1)*VEC_LEN]
# (Here CC ≡ row index, since col_sel drives row-shared activations)

labels_lines = ['sample,filename,true_label,float_pred,total_int8_dot,int8_pred']
for r in range(NPU_ROWS):
    for c in range(NPU_COLS):
        labels_lines[0] += f',dot_tile_{r*NPU_COLS+c:02d}'
labels_lines[0] += '\n'

for i, s in enumerate(all_samples):
    act_q = s['act_q']  # shape (32,)

    # Export 4 column hex files for this sample
    for col_sel in range(NPU_COLS):
        a_slice = act_q[col_sel*VEC_LEN:(col_sel+1)*VEC_LEN]
        to_hex(a_slice, f'psmac_data/input_col{col_sel}_sample{i:02d}.hex')

    # Save xray thumbnail
    xray = Image.open(s['path']).convert('L').resize((128, 128))
    xray.save(f'psmac_data/xray_{i:02d}.png')

    int8_pred = 1 if s['total_dot'] > 0 else 0
    tile_str  = ','.join(str(d) for d in s['tile_dots'])
    labels_lines.append(
        f"{i:02d},input_col0-3_sample{i:02d}.hex,{s['label']},{s['pred']:.4f},{s['total_dot']},{int8_pred},{tile_str}\n"
    )
    print(f"  [{i:02d}] {'PNEUMONIA' if s['label'] else 'NORMAL':9s}  total_dot={s['total_dot']:9d}  "
          f"tiles={[d for d in s['tile_dots']]}  pred={'PNEUMONIA' if int8_pred else 'NORMAL'}")

with open('psmac_data/labels.csv', 'w') as f:
    f.writelines(labels_lines)

# ── Metrics ───────────────────────────────────────────────────────────────────
tp = sum(1 for s in all_samples if s['label']==1 and s['total_dot']>0)
tn = sum(1 for s in all_samples if s['label']==0 and s['total_dot']<=0)
fp = sum(1 for s in all_samples if s['label']==0 and s['total_dot']>0)
fn = sum(1 for s in all_samples if s['label']==1 and s['total_dot']<=0)
acc  = (tp+tn)/(tp+tn+fp+fn)
prec = tp/(tp+fp) if (tp+fp)>0 else 0
rec  = tp/(tp+fn) if (tp+fn)>0 else 0
f1   = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0

with open('psmac_data/metrics.txt', 'w') as f:
    f.write(f'TP={tp}\nTN={tn}\nFP={fp}\nFN={fn}\n')
    f.write(f'Accuracy={acc*100:.1f}\nPrecision={prec*100:.1f}\n')
    f.write(f'Recall={rec*100:.1f}\nF1={f1*100:.1f}\n')
    f.write(f'\n# NPU mapping\n')
    f.write(f'NPU_ROWS={NPU_ROWS}\nNPU_COLS={NPU_COLS}\nNPU_TILES={NPU_TILES}\nVEC_LEN={VEC_LEN}\n')

print(f'\nMetrics: Acc={acc*100:.1f}% Prec={prec*100:.1f}% Rec={rec*100:.1f}% F1={f1*100:.1f}%')
print(f'\nnpu_4_4 instantiation parameters:')
print(f'  vec_len = {VEC_LEN}  (pass to .vec_len port)')
print(f'  mode    = 2\'b10  (INT8)')
print(f'  Tiles 0-3   → row0 activations[0:8]')
print(f'  Tiles 4-7   → row1 activations[8:16]')
print(f'  Tiles 8-11  → row2 activations[16:24]')
print(f'  Tiles 12-15 → row3 activations[24:32]')

In [ ]:
# ── CELL 6: Verify hex files match expected npu_ctrl load sequence ─────────────
# This cell validates that the exported files are consistent with how
# npu_ctrl sequences LOAD_W and LOAD_X phases.

print('=== LOAD_W phase validation ===')
print(f'npu_ctrl will iterate: tile_sel 0→{NPU_TILES-1}, load_addr 0→{VEC_LEN-1}')
for tile_idx in range(NPU_TILES):
    fname = f'psmac_data/weights_tile_{tile_idx:02d}.hex'
    lines = open(fname).readlines()
    assert len(lines) == VEC_LEN, f'Tile {tile_idx}: expected {VEC_LEN} lines, got {len(lines)}'
print(f'✓ All {NPU_TILES} weight files have {VEC_LEN} entries each')

print(f'\n=== LOAD_X phase validation ===')
print(f'npu_ctrl will iterate: col_sel 0→{NPU_COLS-1}, load_addr 0→{VEC_LEN-1}')
for col_sel in range(NPU_COLS):
    fname0 = f'psmac_data/input_col{col_sel}_sample00.hex'
    lines = open(fname0).readlines()
    assert len(lines) == VEC_LEN, f'Col {col_sel}: expected {VEC_LEN} lines, got {len(lines)}'
print(f'✓ All {NPU_COLS} column activation files have {VEC_LEN} entries each')

print(f'\n=== result_sel readback ===')
print('After npu_valid goes high, read result_out for each tile:')
for tile_idx in range(NPU_TILES):
    r = tile_idx // NPU_COLS
    c = tile_idx  % NPU_COLS
    print(f'  result_sel={tile_idx:2d} → tile[row={r},col={c}] (activation slice [{r*VEC_LEN}:{(r+1)*VEC_LEN}])')

print('\nSum all 4 row results → final dot product')
print('Sign of sum → classification: >0 = PNEUMONIA, ≤0 = NORMAL')

In [ ]:
# ── CELL 7: Zip and download ──────────────────────────────────────────────────
import zipfile, pathlib
from google.colab import files

with zipfile.ZipFile('psmac_data.zip','w') as z:
    for f in pathlib.Path('psmac_data').iterdir():
        z.write(str(f), f'psmac_data/{f.name}')

files.download('psmac_data.zip')
print('Downloaded psmac_data.zip')
print(f'\nContents summary:')
print(f'  weights_tile_00.hex .. weights_tile_15.hex  ({NPU_TILES} files, {VEC_LEN} bytes each)')
print(f'  input_col0_sample00.hex .. input_col3_sample39.hex  ({NPU_COLS*40} files, {VEC_LEN} bytes each)')
print(f'  xray_00.png .. xray_39.png  (128×128 thumbnails)')
print(f'  labels.csv, metrics.txt')

In [ ]:
# ── CELL 8: Visualizer — npu_4_4 tile heat map ───────────────────────────────
# Shows which tiles contributed most to classification for a given sample.
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

SAMPLE_IDX = 0   # ← change this to inspect a different sample

s = all_samples[SAMPLE_IDX]
label_str = 'PNEUMONIA' if s['label'] else 'NORMAL'
pred_str  = 'PNEUMONIA' if s['total_dot'] > 0 else 'NORMAL'

# Tile dot products → 4×4 grid
tile_grid = np.array(s['tile_dots']).reshape(NPU_ROWS, NPU_COLS).astype(float)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle(f'Sample {SAMPLE_IDX:02d} | True: {label_str} | INT8 Pred: {pred_str} | Float pred: {s["pred"]:.3f}',
             fontsize=13, fontweight='bold')

# ── Panel 1: X-ray ──────────────────────────────────────────────────────────
xray = np.array(Image.open(f'psmac_data/xray_{SAMPLE_IDX:02d}.png').convert('L'))
axes[0].imshow(xray, cmap='gray')
axes[0].set_title('Chest X-Ray (64→128px)')
axes[0].axis('off')

# ── Panel 2: npu_4_4 tile dot product heat map ──────────────────────────────
cmap = 'RdYlGn' if s['total_dot'] > 0 else 'RdYlGn_r'
im = axes[1].imshow(tile_grid, cmap=cmap, aspect='auto')
axes[1].set_title(f'npu_4_4 Tile Dot Products\n(total = {s["total_dot"]})')
axes[1].set_xlabel('col_sel (0-3)')
axes[1].set_ylabel('row (0-3)')
for r in range(NPU_ROWS):
    for c in range(NPU_COLS):
        axes[1].text(c, r, f't{r*NPU_COLS+c:02d}\n{int(tile_grid[r,c])}',
                     ha='center', va='center', fontsize=8,
                     color='black' if abs(tile_grid[r,c]) < tile_grid.max()*0.7 else 'white')
plt.colorbar(im, ax=axes[1])
axes[1].set_xticks(range(NPU_COLS))
axes[1].set_yticks(range(NPU_ROWS))
axes[1].set_xticklabels([f'col{c}\nact[{c*VEC_LEN}:{(c+1)*VEC_LEN}]' for c in range(NPU_COLS)], fontsize=7)
axes[1].set_yticklabels([f'row{r}\nw[{r*VEC_LEN}:{(r+1)*VEC_LEN}]' for r in range(NPU_ROWS)], fontsize=7)

# ── Panel 3: Per-row accumulated sum ─────────────────────────────────────────
row_sums = tile_grid.sum(axis=1)
colors   = ['#e74c3c' if v < 0 else '#2ecc71' for v in row_sums]
bars     = axes[2].barh(range(NPU_ROWS), row_sums, color=colors)
axes[2].set_title('Row-wise Sum\n(accumulator output)')
axes[2].set_xlabel('Accumulated dot product')
axes[2].set_yticks(range(NPU_ROWS))
axes[2].set_yticklabels([f'Row {r} (tiles {r*NPU_COLS}-{r*NPU_COLS+3})' for r in range(NPU_ROWS)])
axes[2].axvline(0, color='black', linewidth=1)
for bar, val in zip(bars, row_sums):
    axes[2].text(val + (max(abs(row_sums))*0.02), bar.get_y()+bar.get_height()/2,
                 f'{int(val)}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(f'psmac_data/viz_sample_{SAMPLE_IDX:02d}.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: psmac_data/viz_sample_{SAMPLE_IDX:02d}.png')

In [ ]:
# ── CELL 9: Overall metrics summary across all 40 samples ────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('PSMAC npu_4_4 INT8 Inference — Test Set Summary', fontsize=13, fontweight='bold')

# ── Confusion matrix ─────────────────────────────────────────────────────────
cm = np.array([[tn, fp], [fn, tp]])
axes[0].imshow(cm, cmap='Blues')
axes[0].set_title(f'Confusion Matrix\nAcc={acc*100:.1f}% F1={f1*100:.1f}%')
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(cm[i,j]), ha='center', va='center', fontsize=16, fontweight='bold')
axes[0].set_xticks([0,1]); axes[0].set_yticks([0,1])
axes[0].set_xticklabels(['Pred NORMAL','Pred PNEUMONIA'])
axes[0].set_yticklabels(['True NORMAL','True PNEUMONIA'])

# ── Dot product distribution ─────────────────────────────────────────────────
dots_p = [s['total_dot'] for s in all_samples if s['label']==1]
dots_n = [s['total_dot'] for s in all_samples if s['label']==0]
axes[1].hist(dots_p, bins=15, alpha=0.7, color='#e74c3c', label='PNEUMONIA')
axes[1].hist(dots_n, bins=15, alpha=0.7, color='#3498db', label='NORMAL')
axes[1].axvline(0, color='black', linestyle='--', label='Decision boundary')
axes[1].set_title('INT8 Total Dot Product Distribution')
axes[1].set_xlabel('Sum of 4 tile dot products')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.savefig('psmac_data/metrics_summary.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'TP={tp} TN={tn} FP={fp} FN={fn}')
print(f'Acc={acc*100:.1f}%  Prec={prec*100:.1f}%  Rec={rec*100:.1f}%  F1={f1*100:.1f}%')